# Accessing Data

In [1]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [2]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

## Holiday Function

In [3]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Creating new csv file
- One row per day
- 30 days before the holiday
- the holiday itself
- 30 days after the holiday
- So 61 rows per holiday × year × station.
----------------------------------
- For each day, the function will compute:
- Block‑level mean, standard deviation, and variance of the relative‑rank values for that day’s 24‑hour profile.
---------------------------------
- Using your time blocks:
- 04–10
- 10–15
- 15–20
- 20–24
- 00–04
---------------------------------
- Using your 2‑year forward‑looking ranking window (Y + Y+1)

In [5]:
import pandas as pd
import numpy as np
import os

def compute_two_year_daily_relative_rank_csv(
    demand,
    holiday_lib,
    window_days=30,
    blocks=None,
    out_csv="full_nsw_relative_rank.csv"
):
    """
    Computes daily block-level mean, std, and variance of relative ranks
    for each day in ±window_days around each holiday, for each year and station.
    Uses a forward-looking 2-year ranking window (year Y + year Y+1).
    Saves results to CSV in long-form (one row per day).
    """

    if blocks is None:
        raise ValueError("You must supply a dictionary of time blocks.")

    demand.index = pd.to_datetime(demand.index)

    # Hourly mean demand
    hourly = demand.resample("h").mean()

    # Use ALL stations in the dataframe
    stations = [c for c in hourly.columns if c not in ["date", "hour"]]

    rows = []

    # Loop holidays
    for holiday_name, func in holiday_lib.items():

        # Loop years
        for year in range(2004, 2018):

            ref_date = func(year)

            # -------------------------------
            # 2-YEAR FORWARD-LOOKING POOL
            # -------------------------------
            two_year_start = pd.Timestamp(f"{year}-01-01")
            two_year_end   = pd.Timestamp(f"{year+1}-12-31")

            pool = hourly.loc[two_year_start:two_year_end]
            if pool.empty:
                continue

            # -------------------------------
            # Build ±window_days window
            # -------------------------------
            start = ref_date - pd.Timedelta(days=window_days)
            end   = ref_date + pd.Timedelta(days=window_days)

            window = pool.loc[start:end].copy()
            if window.empty:
                continue

            window["date"] = window.index.date
            window["hour"] = window.index.hour

            # -------------------------------
            # Compute relative rank per hour
            # -------------------------------
            for station in stations:

                if station not in window.columns:
                    continue

                w = window.copy()

                w["rank"] = w.groupby("hour")[station].rank(method="average")
                n_days = w.groupby("hour")["date"].transform("nunique")
                w["relative_rank"] = w["rank"] / n_days

                # -------------------------------
                # Compute daily block stats
                # -------------------------------
                for day in sorted(w["date"].unique()):

                    day_mask = w["date"] == day
                    day_rr = w.loc[day_mask, "relative_rank"]

                    # Must have 24 hours
                    if day_rr.shape[0] != 24:
                        continue

                    # Reindex to 0–23 hours
                    day_rr.index = w.loc[day_mask, "hour"]

                    block_stats = {}

                    for block_name, hours in blocks.items():
                        values = day_rr.loc[list(hours)]

                        block_stats[f"{block_name}_mean"] = values.mean()
                        block_stats[f"{block_name}_std"] = values.std()
                        block_stats[f"{block_name}_var"] = values.var()

                    rows.append({
                        "holiday": holiday_name,
                        "year": year,
                        "station": station,
                        "date": pd.Timestamp(day),
                        "is_holiday": (pd.Timestamp(day).date() == ref_date.date()),
                        **block_stats
                    })

    # Convert to DataFrame
    df = pd.DataFrame(rows)

    # Round numeric columns
    df = df.round(4)

    # Ensure directory exists
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)

    # Save CSV
    df.to_csv(out_csv, index=False)

    return df


In [6]:
blocks = {
    "04_10": range(4, 10),
    "10_15": range(10, 15),
    "15_20": range(15, 20),
    "20_24": range(20, 24),
    "00_04": range(0, 4),
}


In [7]:
df = compute_two_year_daily_relative_rank_csv(
    demand=demand,
    holiday_lib=HOLIDAYS_VIC,
    blocks=blocks,
    out_csv="/home/565/pv3484/aus_substation_electricity/figures/full_nsw_relative_rank.csv"
)
